# Model Persistence and Deployment

A model is only useful when it can make predictions outside the notebook. This notebook covers the
last mile:

- training a reproducible pipeline,
- saving and loading the model with `joblib`,
- packaging metadata (feature names, version, metrics),
- serving predictions through a FastAPI endpoint,
- validating inputs and testing the API in-process,
- monitoring, versioning, and security considerations.

## Learning objectives

1. Save and load a complete preprocessing + model pipeline.
2. Bundle metadata with the artifact for reproducibility.
3. Define a typed prediction API with FastAPI and Pydantic.
4. Test the API without starting a separate server.
5. Plan versioning, monitoring, and safe secret handling.

## 1. Concepts

### What to persist

Save the **entire pipeline**, not just the estimator. The pipeline contains the scaler, imputer,
encoder, and model, so new data is transformed consistently.

A useful artifact bundle contains:

- the fitted pipeline,
- the feature names and their order,
- the model version,
- training metrics,
- the library versions used,
- a timestamp.

### Serving options

- **Batch**: score a file on a schedule.
- **Real-time API**: FastAPI, Flask, or a cloud endpoint.
- **Streaming**: consume events and score in a pipeline.
- **Edge**: export to ONNX or a mobile format.

### Production concerns

- Input validation and schema checks.
- Logging without storing sensitive raw data.
- Monitoring for drift, latency, and error rates.
- Versioned artifacts and rollback plans.
- Authentication, rate limiting, and HTTPS.
- Secrets from environment variables or a secret manager, never from source code.

## 2. Setup

In [1]:
import json
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 3. Train a reproducible pipeline

In [2]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])
pipeline.fit(X_train, y_train)

probabilities = pipeline.predict_proba(X_test)[:, 1]
predictions = pipeline.predict(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, predictions)),
    "roc_auc": float(roc_auc_score(y_test, probabilities)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
}
metrics

{'accuracy': 0.9824561403508771,
 'roc_auc': 0.9953703703703703,
 'n_train': 455,
 'n_test': 114}

## 4. Save and load the artifact bundle

We use a temporary directory so the notebook does not leave files behind. In a real project, write to
a versioned artifact location such as `models/breast_cancer/1.0.0/model.joblib`.

In [3]:
artifact = {
    "pipeline": pipeline,
    "feature_names": X.columns.tolist(),
    "target_names": data.target_names.tolist(),
    "model_version": "1.0.0",
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "metrics": metrics,
}

with tempfile.TemporaryDirectory() as tmpdir:
    artifact_path = Path(tmpdir) / "model.joblib"
    joblib.dump(artifact, artifact_path)
    print("Artifact size (KB):", round(artifact_path.stat().st_size / 1024, 1))

    loaded = joblib.load(artifact_path)
    reloaded_predictions = loaded["pipeline"].predict(X_test)
    print("Predictions match after reload:", bool(np.array_equal(predictions, reloaded_predictions)))
    print("Loaded version:", loaded["model_version"])
    print("Feature count:", len(loaded["feature_names"]))

Artifact size (KB): 3.9
Predictions match after reload: True
Loaded version: 1.0.0
Feature count: 30


## 5. Batch prediction

Batch scoring is the simplest deployment pattern. Validate the schema before calling the model.

In [4]:
batch = X_test.head(5).copy()
missing = set(pipeline.feature_names_in_) - set(batch.columns)
extra = set(batch.columns) - set(pipeline.feature_names_in_)
print("Missing columns:", missing)
print("Unexpected columns:", extra)

batch_probabilities = pipeline.predict_proba(batch)[:, 1]
batch_predictions = pipeline.predict(batch)
pd.DataFrame({
    "prediction": batch_predictions,
    "probability": batch_probabilities.round(4),
}).head()

Missing columns: set()
Unexpected columns: set()


,prediction,probability
0,0,0.0000
1,1,1.0000
2,0,0.0064
3,1,0.5335
4,0,0.0000


## 6. A typed FastAPI service

The app below is defined inside the notebook for demonstration. In a real deployment it would live in
`app.py` and run with:

```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

The request body contains the 30 features in the same order used during training. A production API
should use named fields or a schema service so column order is explicit.

In [5]:
app = FastAPI(title="Breast Cancer Classifier", version="1.0.0")

class PredictionRequest(BaseModel):
    features: list[float] = Field(
        ...,
        min_length=len(artifact["feature_names"]),
        max_length=len(artifact["feature_names"]),
        description="Feature values in the training column order.",
    )

class PredictionResponse(BaseModel):
    prediction: int
    label: str
    probability: float
    model_version: str

@app.get("/health")
def health() -> dict:
    return {"status": "ok", "model_version": artifact["model_version"]}

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest) -> PredictionResponse:
    row = np.asarray(request.features, dtype=float).reshape(1, -1)
    probability = float(pipeline.predict_proba(row)[0, 1])
    prediction = int(probability >= 0.5)
    label = data.target_names[prediction]
    return PredictionResponse(
        prediction=prediction,
        label=str(label),
        probability=round(probability, 4),
        model_version=artifact["model_version"],
    )

print("FastAPI app defined")

FastAPI app defined


### Test the API in-process

In [6]:
client = TestClient(app)

health_response = client.get("/health")
print("Health:", health_response.status_code, health_response.json())

payload = {"features": X_test.iloc[0].tolist()}
prediction_response = client.post("/predict", json=payload)
print("Predict:", prediction_response.status_code, prediction_response.json())

bad_payload = {"features": [1.0, 2.0]}
bad_response = client.post("/predict", json=bad_payload)
print("Validation error status:", bad_response.status_code)

Health: 200 {'status': 'ok', 'model_version': '1.0.0'}
Predict: 200 {'prediction': 0, 'label': 'malignant', 'probability': 0.0, 'model_version': '1.0.0'}
Validation error status: 422


The equivalent request with `curl`:

```bash
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.0787,
                    1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
                    25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]}'
```

## 7. Docker sketch

A minimal container for the API:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py model.joblib ./
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

Do not bake secrets into the image. Pass configuration through environment variables or a secret
manager.

## 8. Monitoring and lifecycle

- Track request count, latency, error rate, and prediction distribution.
- Monitor input drift (feature distributions) and output drift (prediction rates).
- Store model version with every prediction for auditability.
- Define a retraining trigger and a rollback procedure.
- Log only what is necessary; avoid raw personal data.
- Test the service with contract tests and a small canary before full rollout.

## Key takeaways

- Persist the whole pipeline, not just the estimator.
- Bundle metadata so predictions can be reproduced and audited.
- Validate inputs and return typed responses.
- Test the API in-process before deploying it.
- Treat monitoring, versioning, and security as part of the model, not an afterthought.

## Exercises

1. Add `/predict_batch` that accepts a list of feature rows.
2. Add named feature fields with Pydantic and validate ranges.
3. Save the artifact to a versioned path and write a `metadata.json` next to it.
4. Add a test for a missing feature and for a non-numeric value.
5. Containerise the API and run it locally.
6. Describe a monitoring dashboard you would build for this service.